# Part 02 — Transformer Architecture: Deep Technical Walkthrough

**Based on:** [bbycroft.net/llm](https://bbycroft.net/llm) — the best interactive LLM visualizer  
**Reference model:** GPT-2 Small (85M params)  
**Goal:** Follow a sentence **"the cat sat"** token-by-token through every matrix operation inside a transformer

---

### What you'll understand after this notebook:
1. Exact tensor shapes at every layer
2. How Q, K, V attention works with real numbers
3. Why residual streams and layer norm matter
4. How causal masking prevents "cheating" in autoregressive generation
5. How logits become predicted words

## GPT-2 Small — Architecture at a Glance

Before diving deep, here are the **exact hyperparameters** of GPT-2 Small.  
Every shape you see below traces back to these numbers.

```
GPT-2 Small Hyperparameters
─────────────────────────────────────────────
vocab_size   = 50,257   # BPE tokens (subwords)
d_model      = 768      # "width" of the model
n_heads      = 12       # attention heads per layer
d_head       = 64       # d_model / n_heads = 768/12
d_ff         = 3,072    # feed-forward hidden dim (4 × d_model)
n_layers     = 12       # transformer blocks stacked
max_ctx      = 1,024    # max tokens the model sees at once
─────────────────────────────────────────────
Total params ≈ 85 million
```

### The Complete Data Flow (shapes shown for T=3 tokens):

```
Input text:  "the cat sat"
             ↓ tokenize
Token IDs:   [the=262, cat=3797, sat=3332]   shape: [T]  = [3]
             ↓ embedding lookup  (W_E: 50257×768)
Token embs:  shape: [T, d_model] = [3, 768]
             ↓ + positional encoding (W_P: 1024×768)
x₀:          shape: [T, d_model] = [3, 768]   ← "residual stream" starts here
             ↓ ×12 transformer blocks (each block below)
─────── Transformer Block (×12) ───────
x = x + Attn(LayerNorm(x))      # multi-head self-attention
x = x + MLP(LayerNorm(x))       # feed-forward network
────────────────────────────────────────
             ↓ final LayerNorm
             ↓ unembed  (W_E^T: 768×50257)
Logits:      shape: [T, vocab_size] = [3, 50257]
             ↓ softmax on last token
Probs:       shape: [vocab_size] = [50257]   → argmax → next token predicted
```


You’re **not** getting a black box — under the hood, it’s made of **PyTorch layers**

## **Key Technical Terms Simplified:**

- **Token**: A piece of text (word, subword, or character)
- **Embedding**: Converting text into numbers
- **Attention**: How much focus to put on different parts of the input
- **Head**: One of multiple parallel attention mechanisms
- **Layer**: One complete processing step
- **Parameters**: The learned numbers that make the model work
  
## Core Architecture – **The Transformer as an Orchestra**

Imagine the Transformer as a **grand concert hall** where a group of musicians (tokens) prepare and perform together.

---

### **1. The Musicians Arrive – Input Embeddings**

- **Analogy:** Each musician brings their own instrument and playing style. That’s their **unique musical style**.
- **Simple terms**: Convert words into numbers that computers can understand
- **Code**: Each word becomes a vector of 512 numbers that captures its "meaning"
- **Example**: "dog" and "puppy" get similar number patterns because they mean similar things

---

### **2. Seating Arrangement – Positional Encoding**

- **Analogy:** The conductor assigns each musician a specific seat — violins front-left, trumpets in back-right — so timing and harmony work.
- **Simple terms**: Tell the computer where each word sits in the sentence
- **Code**: Add special position patterns to each word's numbers. A **sine/cosine positional encoding** or a **learned position vector** is added to embeddings to preserve word order.
- **Example**: "Cat chased mouse" vs "Mouse chased cat" - same words, different positions, different meanings. 

---

### **3. Listening to Each Other – Self-Attention**

- **Analogy:** Each musician listens to others to decide how to play. The violin might focus on cello harmony or flute melody cues.
- **Simple terms**: Each word "looks at" all other words to understand context
- **Code**: Calculate how much each word should pay attention to every other word
  This is calculated by:

  ```python
  Attention(Q, K, V) = softmax(QKᵀ / √dₖ) V

  
  ```
  
  * where Q (Query), K (Key), V (Value) come from the embeddings.

  ```

  ```

- **Example**: In `"The bank of the river"`, `"bank"` pays more attention to `"river"` than `"money"`

---

### **4. Different Ears – Multi-Head Attention**

- **Analogy:** Musicians listen with multiple “ears” — one for rhythm, one for pitch, one for emotion — and blend them.
- **Simple terms**: Look at relationships in multiple ways simultaneously
- **Code**: Split attention into 8 different "heads" that each focus on different patterns. One attention head might track subject-verb agreement, another tracks long-range context, another tracks named entities.
- **Example**: One head focuses on grammar, another on meaning, another on long-distance relationships

---

### **5. Personal Practice – Feed Forward Network**

- **Analogy:** After listening, each musician rehearses their tricky solo privately before rejoining the orchestra.
- **Simple terms**: Each word processes its information independently
- **Code**: Expand each word's representation, apply transformations, then compress back

  A position-wise feed-forward network:


  ```python
  FFN(x) = ReLU(xW₁ + b₁)W₂ + b₂
  ```

  ```

  ```
  * applies the same transformation to each token independently.

---

### **6. Staying in Tune – Residual Connections + Layer Normalization**

- **Analogy:** The conductor ensures no one plays out of tune or too loud, blending original notes with refinements.
- **Simple terms**: Keep the original information while adding improvements
- **Code**: Add the output back to the input, and normalize to prevent things from getting too big/small

  ```python
  x = LayerNorm(x + Attention(x))
  ```
  ```

  ```
  * This keeps gradients stable during deep training and prevents information loss.

---

### **7. Multiple Rehearsals – Encoder Stack**

- **Analogy:** The orchestra rehearses multiple times, each pass improving timing and dynamics.
- **Simple terms**: Repeat the whole process multiple times to get better understanding
- **Code**: Stack 6 identical layers, each one refining the representation further. BERT-base has 12 encoder layers; BERT-large has 24. Each layer learns progressively richer representations.

---

### **8. Final Performance – Output Projection**

- **Analogy:** The concert is broadcast to the audience, turning all that coordination into actual music.
- **Simple terms**: Convert the final understanding back into word probabilities
- **Code**: Transform the final hidden states into probabilities over the entire vocabulary. Final hidden states are multiplied by the **output embedding matrix** and passed through a **softmax** to choose the most likely token
- **Example**: Predict what word comes next in "The capital of France is ___" → "Paris"

---

## **Summary Table with Examples**

| Transformer Part     | Orchestra Analogy               | Purpose                             | Technical Example                           |
| -------------------- | ------------------------------- | ----------------------------------- | ------------------------------------------- |
| Embedding            | Musician’s unique sound         | Represent words numerically         | `"dog"` → vector of size 768 / 12,288       |
| Positional Encoding  | Seat on stage                   | Keep order/timing info              | Sinusoidal encoding for index 0, 1, 2…      |
| Self-Attention       | Listening to others             | Contextual relationships            | `"bank"` attends to `"river"` not `"money"` |
| Multi-Head Attention | Listening with different “ears” | Capture multiple aspects of context | Head 1 → syntax, Head 2 → long-range deps   |
| Feed Forward         | Individual practice             | Refine each musician’s part         | FFN with ReLU applied to each token         |
| Residual + LayerNorm | Conductor’s tuning check        | Preserve and balance info           | `x = LayerNorm(x + sublayer(x))`            |
| Encoder Stack        | Multiple rehearsals             | Progressive refinement              | BERT-base: 12 layers                        |s
| Output Layer         | Final performance               | Convert back to words               | Softmax over vocab to pick `"Paris"`        |


### Code Snippet:

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# ============================================================================
# TRANSFORMER AS AN ORCHESTRA 🎼
# ============================================================================

class TransformerOrchestra(nn.Module):
    """
    The complete orchestra - our transformer model
    Each component represents a different part of the musical performance
    """
    def __init__(self, vocab_size=10000, d_model=512, n_heads=8, n_layers=6, max_seq_len=1000):
        super().__init__()
        
        # 1. MUSICIANS ARRIVE - Input Embeddings
        self.token_embedding = MusicianEmbedding(vocab_size, d_model)
        
        # 2. SEATING ARRANGEMENT - Positional Encoding  
        self.positional_encoding = SeatingArrangement(d_model, max_seq_len)
        
        # 3-7. MULTIPLE REHEARSALS - Encoder Stack
        self.encoder_layers = nn.ModuleList([
            OrchestraLayer(d_model, n_heads) for _ in range(n_layers)
        ])
        
        # 8. FINAL PERFORMANCE - Output Projection
        self.output_projection = FinalPerformance(d_model, vocab_size)
        
        self.d_model = d_model
        
    def forward(self, x):
        # Musicians arrive with their instruments
        x = self.token_embedding(x)
        
        # Everyone takes their assigned seats
        x = self.positional_encoding(x)
        
        # Multiple rehearsals (each layer)
        for layer in self.encoder_layers:
            x = layer(x)
            
        # Final performance for the audience
        return self.output_projection(x)


# ============================================================================
# 1. THE MUSICIANS ARRIVE – INPUT EMBEDDINGS 🎻🎺🥁
# ============================================================================

class MusicianEmbedding(nn.Module):
    """
    Each word/token is like a musician with their unique playing style.
    We convert words into high-dimensional vectors that capture their 'musical essence'
    
    Think: 'dog' and 'puppy' should have similar 'musical styles' (close vectors)
    """
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model
        
    def forward(self, x):
        # Scale embeddings by sqrt(d_model) for better training stability
        return self.embedding(x) * math.sqrt(self.d_model)

# Example usage:
def demonstrate_embeddings():
    print("🎻 MUSICIANS ARRIVING - INPUT EMBEDDINGS")
    print("=" * 50)
    
    vocab_size = 1000
    d_model = 512
    
    # Create embedding layer
    musician_embedding = MusicianEmbedding(vocab_size, d_model)
    
    # Sample tokens (imagine: [the, cat, sat, on, mat])
    tokens = torch.tensor([1, 45, 123, 67, 234])
    
    # Convert to embeddings
    embeddings = musician_embedding(tokens)
    
    print(f"Input tokens: {tokens}")
    print(f"Embedding shape: {embeddings.shape}")  # [5, 512]
    print(f"Each token becomes a {d_model}-dimensional vector representing its 'musical style'")
    print(f"Token 1 embedding (first 5 dims): {embeddings[0, :5]}")
    print()


# ============================================================================
# 2. SEATING ARRANGEMENT – POSITIONAL ENCODING 🪑
# ============================================================================

class SeatingArrangement(nn.Module):
    """
    The conductor assigns each musician a specific seat so timing works properly.
    Position matters: 'The cat sat' vs 'Sat the cat' have different meanings!
    
    We add positional information to embeddings using sine/cosine waves
    """
    def __init__(self, d_model, max_seq_len=1000):
        super().__init__()
        
        # Create a matrix to hold positional encodings
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        
        # Create the sinusoidal pattern
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dimensions
        
        # Register as buffer (not a parameter, but part of model state)
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        # Add positional encoding to embeddings
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

def demonstrate_positional_encoding():
    print("🪑 SEATING ARRANGEMENT - POSITIONAL ENCODING")
    print("=" * 50)
    
    d_model = 512
    max_seq_len = 100
    
    seating = SeatingArrangement(d_model, max_seq_len)
    
    # Sample embeddings for 5 tokens
    embeddings = torch.randn(1, 5, d_model)  # [batch_size, seq_len, d_model]
    
    # Add positional information
    positioned_embeddings = seating(embeddings)
    
    print(f"Original embeddings shape: {embeddings.shape}")
    print(f"After adding positions: {positioned_embeddings.shape}")
    print("Each token now knows its position in the sequence!")
    print(f"Position encoding for pos 0 (first 5 dims): {seating.pe[0, 0, :5]}")
    print(f"Position encoding for pos 1 (first 5 dims): {seating.pe[0, 1, :5]}")
    print()


# ============================================================================
# 3. LISTENING TO EACH OTHER – SELF-ATTENTION 🎵
# ============================================================================

class ListeningToOthers(nn.Module):
    """
    Each musician listens to others to decide how to play their part.
    In 'The bank of the river', 'bank' should pay more attention to 'river' than 'money'
    
    This is the heart of the transformer: attention mechanism!
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.sqrt_d_model = math.sqrt(d_model)
        
        # Create Q, K, V transformation matrices
        self.query_transform = nn.Linear(d_model, d_model)
        self.key_transform = nn.Linear(d_model, d_model)
        self.value_transform = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        
        # Transform input into Query, Key, Value
        Q = self.query_transform(x)  # What am I looking for?
        K = self.key_transform(x)    # What do I have to offer?
        V = self.value_transform(x)  # What is my actual content?
        
        # Calculate attention scores
        # Each token asks: "How much should I pay attention to every other token?"
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / self.sqrt_d_model
        
        # Convert scores to probabilities (softmax)
        attention_weights = F.softmax(attention_scores, dim=-1)
        
        # Apply attention to values
        attended_output = torch.matmul(attention_weights, V)
        
        return attended_output, attention_weights

def demonstrate_attention():
    print("🎵 LISTENING TO EACH OTHER - SELF-ATTENTION")
    print("=" * 50)
    
    d_model = 512
    seq_len = 5
    batch_size = 1
    
    # Sample positioned embeddings
    x = torch.randn(batch_size, seq_len, d_model)
    
    attention = ListeningToOthers(d_model)
    output, weights = attention(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Attention weights shape: {weights.shape}")  # [batch, seq_len, seq_len]
    print("\nAttention matrix (how much each token attends to others):")
    print("Rows = tokens asking, Columns = tokens being attended to")
    print(weights[0].detach().numpy().round(3))
    print()


# ============================================================================
# 4. DIFFERENT EARS – MULTI-HEAD ATTENTION 👂👂👂
# ============================================================================

class DifferentEars(nn.Module):
    """
    Musicians listen with multiple 'ears':
    - One ear for rhythm patterns
    - One ear for harmonic relationships  
    - One ear for melodic themes
    - etc.
    
    Each 'head' captures different types of relationships between tokens
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Dimension per head
        
        # Single linear layers for all heads (more efficient)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
        
    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        
        # Transform and reshape for multiple heads
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Apply attention for each head
        attention_output = self.scaled_dot_product_attention(Q, K, V)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, d_model
        )
        
        return self.W_o(attention_output)
    
    def scaled_dot_product_attention(self, Q, K, V):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        attention_weights = F.softmax(scores, dim=-1)
        return torch.matmul(attention_weights, V)

def demonstrate_multi_head_attention():
    print("👂 DIFFERENT EARS - MULTI-HEAD ATTENTION")
    print("=" * 50)
    
    d_model = 512
    n_heads = 8
    seq_len = 5
    batch_size = 1
    
    x = torch.randn(batch_size, seq_len, d_model)
    
    multi_head_attention = DifferentEars(d_model, n_heads)
    output = multi_head_attention(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Number of attention heads: {n_heads}")
    print(f"Dimension per head: {d_model // n_heads}")
    print(f"Output shape: {output.shape}")
    print("Each head learns to focus on different aspects of relationships!")
    print()


# ============================================================================
# 5. PERSONAL PRACTICE – FEED FORWARD NETWORK 🎼
# ============================================================================

class PersonalPractice(nn.Module):
    """
    After listening to the orchestra, each musician practices their part privately.
    This is a position-wise feed-forward network - each token is processed independently.
    
    Think: expanding the representation, applying non-linearity, then compressing back
    """
    def __init__(self, d_model, d_ff=2048):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, x):
        # Expand, activate, compress
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

def demonstrate_feed_forward():
    print("🎼 PERSONAL PRACTICE - FEED FORWARD NETWORK")
    print("=" * 50)
    
    d_model = 512
    d_ff = 2048
    seq_len = 5
    batch_size = 1
    
    x = torch.randn(batch_size, seq_len, d_model)
    
    ffn = PersonalPractice(d_model, d_ff)
    output = ffn(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Hidden dimension: {d_ff}")
    print(f"Output shape: {output.shape}")
    print("Each token is processed independently through the same network!")
    print()


# ============================================================================
# 6. STAYING IN TUNE – RESIDUAL CONNECTIONS + LAYER NORMALIZATION 🎵
# ============================================================================

class StayingInTune(nn.Module):
    """
    The conductor ensures no one goes off-key and everyone stays balanced.
    
    Residual connections: Keep the original melody while adding refinements
    Layer normalization: Keep all musicians at similar volume levels
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, sublayer_fn):
        # Pre-norm: normalize first, then apply sublayer, then add residual
        return x + self.dropout(sublayer_fn(self.norm(x)))

def demonstrate_residual_norm():
    print("🎵 STAYING IN TUNE - RESIDUAL + LAYER NORM")
    print("=" * 50)
    
    d_model = 512
    seq_len = 5
    batch_size = 1
    
    x = torch.randn(batch_size, seq_len, d_model)
    
    staying_in_tune = StayingInTune(d_model)
    
    # Example sublayer (could be attention or feed-forward)
    def example_sublayer(x):
        return torch.randn_like(x) * 0.1  # Small random transformation
    
    output = staying_in_tune(x, example_sublayer)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Original input mean: {x.mean():.4f}, std: {x.std():.4f}")
    print(f"Normalized mean: {output.mean():.4f}, std: {output.std():.4f}")
    print("Residual connection preserves original information!")
    print()


# ============================================================================
# 7. COMPLETE ORCHESTRA LAYER 🎭
# ============================================================================

class OrchestraLayer(nn.Module):
    """
    One complete rehearsal: attention + feed-forward with proper normalization
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.multi_head_attention = DifferentEars(d_model, n_heads)
        self.feed_forward = PersonalPractice(d_model)
        self.norm1 = StayingInTune(d_model)
        self.norm2 = StayingInTune(d_model)
        
    def forward(self, x):
        # First sublayer: multi-head attention
        x = self.norm1(x, self.multi_head_attention)
        
        # Second sublayer: feed-forward
        x = self.norm2(x, self.feed_forward)
        
        return x


# ============================================================================
# 8. FINAL PERFORMANCE – OUTPUT PROJECTION 🎪
# ============================================================================

class FinalPerformance(nn.Module):
    """
    The orchestra's music is broadcast to the audience.
    Convert internal representations back to vocabulary probabilities.
    """
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)
        
    def forward(self, x):
        return self.linear(x)


# ============================================================================
# COMPLETE DEMONSTRATION 🎉
# ============================================================================

def complete_demonstration():
    print("\n" + "="*60)
    print("🎼 COMPLETE TRANSFORMER ORCHESTRA PERFORMANCE 🎼")
    print("="*60)
    
    # Model parameters
    vocab_size = 1000
    d_model = 512
    n_heads = 8
    n_layers = 6
    max_seq_len = 100
    batch_size = 2
    seq_len = 10
    
    # Create the complete orchestra
    orchestra = TransformerOrchestra(vocab_size, d_model, n_heads, n_layers, max_seq_len)
    
    # Sample input (batch of token sequences)
    input_tokens = torch.randint(0, vocab_size, (batch_size, seq_len))
    
    print(f"🎭 Orchestra Configuration:")
    print(f"   Vocabulary size: {vocab_size:,}")
    print(f"   Model dimension: {d_model}")
    print(f"   Attention heads: {n_heads}")
    print(f"   Number of layers: {n_layers}")
    print(f"   Max sequence length: {max_seq_len}")
    
    print(f"\n🎵 Performance:")
    print(f"   Input shape: {input_tokens.shape}")
    print(f"   Input tokens (first sequence): {input_tokens[0]}")
    
    # Forward pass through the complete model
    with torch.no_grad():
        output = orchestra(input_tokens)
    
    print(f"   Output shape: {output.shape}")  # [batch_size, seq_len, vocab_size]
    print(f"   Output represents probability distribution over {vocab_size} possible next tokens")
    
    # Get the most likely next tokens
    predicted_tokens = torch.argmax(output, dim=-1)
    print(f"   Predicted tokens (first sequence): {predicted_tokens[0]}")
    
    # Count parameters
    total_params = sum(p.numel() for p in orchestra.parameters())
    print(f"\n📊 Orchestra Size: {total_params:,} parameters")
    
    print("\n✨ The orchestra has performed! Each layer refined the musical understanding,")
    print("   and the final output gives us probabilities for what comes next in the sequence.")


# ============================================================================
# RUN ALL DEMONSTRATIONS
# ============================================================================

if __name__ == "__main__":
    print("🎼 TRANSFORMER ARCHITECTURE: THE ORCHESTRA ANALOGY 🎼\n")
    
    # Run individual demonstrations
    demonstrate_embeddings()
    demonstrate_positional_encoding()
    demonstrate_attention()
    demonstrate_multi_head_attention()
    demonstrate_feed_forward()
    demonstrate_residual_norm()
    
    # Complete demonstration
    complete_demonstration()
    
    print("\n" + "="*60)
    print("🎉 PERFORMANCE COMPLETE! 🎉")
    print("="*60)
    print("\nKey takeaways:")
    print("• Each token (word) is like a musician with a unique style (embedding)")
    print("• Position matters - musicians need assigned seats (positional encoding)")  
    print("• Musicians listen to each other to coordinate (self-attention)")
    print("• Multiple 'ears' capture different relationships (multi-head attention)")
    print("• Individual practice refines each part (feed-forward network)")
    print("• The conductor keeps everyone in tune (layer norm + residuals)")
    print("• Multiple rehearsals improve the performance (stacked layers)")
    print("• The final performance is broadcast to the audience (output projection)")

---
## Step 1 — Tokenization & Embedding Lookup

### What is a Token?
A **token** is the basic unit of text the model processes. GPT-2 uses **Byte-Pair Encoding (BPE)** — a subword tokenizer.

```
"the cat sat on the mat"
       ↓ BPE tokenizer
["the", " cat", " sat", " on", " the", " mat"]
       ↓ vocab lookup
[262, 3797, 3332, 319, 262, 2603]
```

- Common words → single tokens (`the` = 262)
- Rare words → split into subwords (`unbelievable` → `un`, `believ`, `able`)
- This lets the model handle any word with a fixed 50,257-token vocabulary

### Embedding Lookup — Converting IDs to Vectors

The model has an **embedding matrix** `W_E` of shape `[50257, 768]`.  
Each row is a learned 768-dim vector for one token.

```
token_id = 3797  ("cat")
                 ↓ index into W_E
embedding = W_E[3797]   →  [0.12, -0.43, 0.87, ..., 0.21]   shape: [768]
```

**Why 768 dimensions?** Each dimension encodes some learned feature — syntax, semantics, morphology. No single dimension has a human-readable meaning, but together they capture the "essence" of the token.

### Positional Encoding — Telling the Model Where Things Are

Attention is **order-agnostic** by design. Without positional info, `"cat chased dog"` and `"dog chased cat"` would look identical.

GPT-2 uses **learned positional embeddings** `W_P` of shape `[1024, 768]`:
```
position 0 → W_P[0]  →  [0.03, -0.11, ..., 0.55]   (learned pattern for position 0)
position 1 → W_P[1]  →  [-0.22, 0.44, ..., 0.01]
position 2 → W_P[2]  →  [0.17, -0.09, ..., -0.33]
```

**Final input to the transformer:**
```
x[i] = W_E[token_id[i]] + W_P[i]     shape: [768] per token
x    = [x[0], x[1], x[2]]             shape: [3, 768] for 3 tokens
```
This is the **residual stream** — a 768-dim information highway that flows through all 12 layers.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ──────────────────────────────────────────────────────────────────────────────
# STEP 1: Tokenization & Embedding — Concrete walkthrough
# ──────────────────────────────────────────────────────────────────────────────

# GPT-2 Small hyperparameters (exact values)
VOCAB_SIZE  = 50257
D_MODEL     = 768
N_HEADS     = 12
D_HEAD      = D_MODEL // N_HEADS      # 64
D_FF        = D_MODEL * 4             # 3072
N_LAYERS    = 12
MAX_CTX     = 1024

print("=" * 60)
print("STEP 1: Tokenization & Embedding Lookup")
print("=" * 60)

# Simulate token IDs for "the cat sat"
# In real GPT-2:  "the" → 262,  " cat" → 3797,  " sat" → 3332
token_ids = torch.tensor([262, 3797, 3332])   # shape: [T=3]
T = token_ids.shape[0]

print(f"\nInput sentence: 'the cat sat'")
print(f"Token IDs:      {token_ids.tolist()}")
print(f"  262  → 'the'")
print(f"  3797 → ' cat'")
print(f"  3332 → ' sat'")

# ── Embedding matrix (W_E): each row is a learned vector ──
torch.manual_seed(42)
W_E = torch.randn(VOCAB_SIZE, D_MODEL) * 0.02    # realistic init scale
W_P = torch.randn(MAX_CTX,    D_MODEL) * 0.01    # positional embeddings

token_embs = W_E[token_ids]                       # shape: [3, 768]
pos_embs   = W_P[:T]                              # shape: [3, 768]

x0 = token_embs + pos_embs                        # shape: [3, 768]  ← residual stream

print(f"\nEmbedding matrix W_E shape: {W_E.shape}   ({VOCAB_SIZE:,} tokens × {D_MODEL} dims)")
print(f"Positional matrix W_P shape: {W_P.shape}  ({MAX_CTX} positions × {D_MODEL} dims)")
print(f"\ntoken_embs shape: {token_embs.shape}  (3 tokens × 768 dims)")
print(f"pos_embs   shape: {pos_embs.shape}  (3 positions × 768 dims)")
print(f"x0         shape: {x0.shape}  ← RESIDUAL STREAM (token + position info)")

print(f"\nFirst 6 values of 'the' embedding:  {token_embs[0, :6].tolist()}")
print(f"First 6 values of 'cat' embedding:  {token_embs[1, :6].tolist()}")
print(f"First 6 values of 'sat' embedding:  {token_embs[2, :6].tolist()}")
print(f"\nNote: 'cat' and 'sat' are very different — they have different token IDs")
print(f"After training, semantically similar words cluster close together in 768D space.")

---
## Step 2 — Layer Normalization: Stabilizing the Residual Stream

Before attention runs, every transformer block applies **Layer Normalization**.  
This is one of the most important (and underappreciated) components.

### The Problem it Solves
As the residual stream accumulates updates from 12 layers, values can grow wildly different in scale:
```
x[0] might have mean=5.3, std=12.1   → "the" representation exploded
x[1] might have mean=0.1, std=0.3    → "cat" representation collapsed
```
Attention scores computed on these would be dominated by whichever token has the largest values. Bad.

### What LayerNorm Does

For each token's 768-dim vector independently:
```
1. Compute mean:  μ = (1/768) Σ xᵢ
2. Compute std:   σ = sqrt((1/768) Σ (xᵢ - μ)²)
3. Normalize:     x̂ᵢ = (xᵢ - μ) / (σ + ε)    ← ε=1e-5 prevents div-by-zero
4. Rescale:       yᵢ = γ · x̂ᵢ + β             ← γ, β are learned parameters
```

After this, every token's vector has **mean=0, std≈1** before going into attention.  
The learned `γ` and `β` let the model re-scale if needed.

### Pre-norm vs Post-norm
```
Original 2017 Transformer:   x = LayerNorm(x + Sublayer(x))   ← post-norm
GPT-2 / Modern:              x = x + Sublayer(LayerNorm(x))   ← pre-norm (more stable)
```
GPT-2 uses **pre-norm** because gradients flow more cleanly through the residual path.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 2: Layer Normalization — Manual implementation + verification
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("STEP 2: Layer Normalization")
print("=" * 60)

# Create an artificial "exploded" residual stream to show why LayerNorm helps
torch.manual_seed(0)
x_bad = x0.clone()
x_bad[0] = x_bad[0] * 50 + 30    # simulate "the" exploding (mean≈30, std≈large)
x_bad[1] = x_bad[1] * 0.01       # simulate "cat" collapsing (mean≈0, std≈tiny)

print("\nBEFORE LayerNorm (simulated exploded residual stream):")
for i, name in enumerate(["the", "cat", "sat"]):
    print(f"  '{name}'  mean={x_bad[i].mean().item():.3f}  std={x_bad[i].std().item():.3f}")

# ── Manual LayerNorm (matches PyTorch's implementation exactly) ──
def layer_norm_manual(x, gamma, beta, eps=1e-5):
    """
    x:     shape [T, D]
    gamma: shape [D]  — learned scale
    beta:  shape [D]  — learned shift
    """
    mean = x.mean(dim=-1, keepdim=True)          # [T, 1]
    var  = x.var(dim=-1, keepdim=True, unbiased=False)  # [T, 1]
    x_hat = (x - mean) / (var + eps).sqrt()      # normalize: [T, D]
    return gamma * x_hat + beta                  # rescale: [T, D]

# Initialize γ=1, β=0 (identity transform — same as untrained LayerNorm)
gamma = torch.ones(D_MODEL)
beta  = torch.zeros(D_MODEL)

x_normed = layer_norm_manual(x_bad, gamma, beta)

print("\nAFTER LayerNorm:")
for i, name in enumerate(["the", "cat", "sat"]):
    print(f"  '{name}'  mean={x_normed[i].mean().item():.6f}  std={x_normed[i].std().item():.4f}")

# ── Verify against PyTorch ──
ln = nn.LayerNorm(D_MODEL)  # γ=1, β=0 by default
x_torch = ln(x_bad)
max_diff = (x_normed - x_torch).abs().max().item()
print(f"\nMax difference from PyTorch LayerNorm: {max_diff:.2e}  (essentially zero ✓)")

print("""
Key insight:
  - LayerNorm operates on the LAST dimension (D_MODEL=768)
  - Each token's 768-dim vector is normalized INDEPENDENTLY
  - This is different from BatchNorm, which normalizes across the batch
  - After LayerNorm, the mean is ~0 and std is ~1 for every token
  - The learned γ and β allow the model to re-scale/shift if needed
""")

---
## Step 3 — Self-Attention: The Heart of the Transformer

This is the key operation. Every token gathers information from every other token.

### The Q, K, V Framework — A Library Analogy

Think of it like a library search:
- **Query (Q)** — "I'm looking for books about rivers"  (what you're searching for)
- **Key (K)**   — "This book is about: rivers, water, nature"  (what each book advertises)
- **Value (V)** — The actual book content  (what you get when you retrieve it)

The attention mechanism computes: *how much does each Query match each Key?*  
Then returns a weighted sum of Values.

### The Math (single head, no batch dimension)

Given input `x` of shape `[T, D]` (T tokens, D=768 dims):

```
Step A — Project to Q, K, V:
  Q = x @ W_Q    shape: [T, d_head]   W_Q: [D, d_head] = [768, 64]
  K = x @ W_K    shape: [T, d_head]   W_K: [768, 64]
  V = x @ W_V    shape: [T, d_head]   W_V: [768, 64]

Step B — Compute raw attention scores (dot product):
  scores = Q @ K.T    shape: [T, T] = [3, 3]
  
  This gives: how much does token i's query match token j's key?
  
  Example for "the cat sat":
           the   cat   sat
  the  [[ 2.1,  0.3, -0.5],    ← how much "the" attends to each token
  cat   [ 0.8,  3.2,  1.1],    ← how much "cat" attends to each token
  sat   [-0.2,  1.4,  2.8]]    ← how much "sat" attends to each token

Step C — Scale (prevent vanishing gradients):
  scores = scores / sqrt(d_head)    = scores / sqrt(64) = scores / 8
  
  WHY? When d_head=64, Q and K vectors have 64 dimensions.
  Their dot product grows as O(sqrt(d_head)), making softmax too peaky.
  Dividing by sqrt(d_head) keeps gradients in a good range.

Step D — Causal mask (decoder-only models like GPT-2):
  Mask future tokens so position i can only attend to positions ≤ i.
  
  Set future positions to -∞ before softmax:
           the   cat   sat
  the  [[ 2.1,  -∞,   -∞ ],    ← "the" can only see itself (position 0)
  cat   [ 0.8,  3.2,  -∞ ],    ← "cat" can see "the" and itself
  sat   [-0.2,  1.4,  2.8]]    ← "sat" can see all three

Step E — Softmax (convert scores → weights that sum to 1):
           the   cat   sat
  the  [[ 1.0,  0.0,  0.0],    ← "the" is forced to only look at itself
  cat   [ 0.17, 0.83, 0.0 ],   ← "cat" mostly looks at itself
  sat   [ 0.07, 0.28, 0.65]]   ← "sat" mostly looks at itself but some "cat"

Step F — Weighted sum of Values:
  output = weights @ V    shape: [T, d_head] = [3, 64]
  
  output[2] = 0.07*V[0] + 0.28*V[1] + 0.65*V[2]
  ("sat"'s new representation mixes info from all visible tokens)
```

### Why Multi-Head? (GPT-2 has 12 heads)

One head can only capture one "type" of relationship. Multiple heads run in parallel:
```
Head 1  (d_head=64): might learn subject-verb relationships
Head 2  (d_head=64): might learn pronoun-antecedent relationships  
Head 3  (d_head=64): might learn adjective-noun relationships
...
Head 12 (d_head=64): might learn positional patterns

All 12 heads run in parallel → concatenate outputs → [T, 12*64] = [T, 768]
Then apply output projection W_O: [T, 768] → [T, 768]
```

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 3: Self-Attention — Full step-by-step with shapes and values
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("STEP 3: Self-Attention Walkthrough (Single Head)")
print("=" * 60)

torch.manual_seed(7)

# Use a smaller d_model for clarity (same math, easier to inspect)
D_DEMO  = 8       # mini d_model (normally 768)
D_HEAD  = 4       # mini d_head  (normally 64)
TOKENS  = ["the", "cat", "sat"]
T       = 3

# Simulated (small) residual stream input
x = torch.randn(T, D_DEMO)
print(f"\nInput x (residual stream after embedding+pos):  shape {x.shape}")
for i, tok in enumerate(TOKENS):
    print(f"  x[{i}] '{tok}': {x[i].tolist()}")

# ─── A. Project to Q, K, V ─────────────────────────────────────────────────
W_Q = torch.randn(D_DEMO, D_HEAD) * 0.3
W_K = torch.randn(D_DEMO, D_HEAD) * 0.3
W_V = torch.randn(D_DEMO, D_HEAD) * 0.3

Q = x @ W_Q    # [3, 4]
K = x @ W_K    # [3, 4]
V = x @ W_V    # [3, 4]

print(f"\nA. Projection to Q, K, V:")
print(f"   W_Q shape: {W_Q.shape}  (D_DEMO={D_DEMO} → D_HEAD={D_HEAD})")
print(f"   Q = x @ W_Q  → shape {Q.shape}")
print(f"   K = x @ W_K  → shape {K.shape}")
print(f"   V = x @ W_V  → shape {V.shape}")

# ─── B. Raw attention scores ────────────────────────────────────────────────
raw_scores = Q @ K.T    # [3, 3]
print(f"\nB. Raw scores = Q @ K.T  → shape {raw_scores.shape}")
print(f"   (each entry [i,j] = how much token i's query matches token j's key)")
print(f"   Raw scores:\n{raw_scores.detach().numpy().round(3)}")

# ─── C. Scale by sqrt(d_head) ───────────────────────────────────────────────
scale  = math.sqrt(D_HEAD)
scaled = raw_scores / scale
print(f"\nC. Scaled scores (÷ sqrt({D_HEAD})={scale:.2f}):")
print(f"   {scaled.detach().numpy().round(3)}")

# ─── D. Causal mask (GPT-2 style: lower-triangular) ────────────────────────
# Position i can only attend to positions 0..i
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()   # upper triangle = True
masked = scaled.masked_fill(causal_mask, float('-inf'))

print(f"\nD. After causal mask (future = -inf):")
print(f"   Mask pattern (True = blocked):\n{causal_mask.numpy()}")
print(f"   Masked scores:\n{masked.detach().numpy().round(3)}")
print(f"   Note: 'the' (pos 0) can only see itself, 'cat' sees 0+1, 'sat' sees all 3")

# ─── E. Softmax → attention weights ────────────────────────────────────────
attn_weights = F.softmax(masked, dim=-1)   # [3, 3]
print(f"\nE. Attention weights after softmax (each row sums to 1.0):")
header = f"{'':>8}" + "".join(f"  {t:>6}" for t in TOKENS)
print(header)
for i, tok in enumerate(TOKENS):
    row = attn_weights[i].detach().numpy()
    vals = "".join(f"  {v:>6.3f}" for v in row)
    print(f"  {tok:>6}: {vals}")
print(f"  Row sums: {attn_weights.sum(dim=-1).tolist()}")

# ─── F. Weighted sum of Values → output ─────────────────────────────────────
output = attn_weights @ V   # [3, 4]
print(f"\nF. Output = attn_weights @ V  → shape {output.shape}")
print(f"   'sat' output = {attn_weights[2,0]:.3f}*V['the'] + "
      f"{attn_weights[2,1]:.3f}*V['cat'] + {attn_weights[2,2]:.3f}*V['sat']")
print(f"   output['sat']: {output[2].detach().tolist()}")
print(f"\n   This is a context-aware representation of 'sat' that has")
print(f"   gathered information from all preceding tokens!")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 3 (continued): Visualize attention weights + softmax temperature effect
# ──────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Attention Weight Patterns & Temperature Effect", fontsize=13, fontweight='bold')

TOKENS = ["the", "cat", "sat"]

# ── Panel 1: Our "the cat sat" attention weights ──────────────────────────
ax = axes[0]
weights_np = attn_weights.detach().numpy()
im = ax.imshow(weights_np, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(TOKENS, fontsize=11)
ax.set_yticklabels(TOKENS, fontsize=11)
ax.set_xlabel("Keys (tokens being attended TO)", fontsize=9)
ax.set_ylabel("Queries (tokens ASKING)", fontsize=9)
ax.set_title("Causal Attention\n'the cat sat'", fontsize=11)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{weights_np[i,j]:.2f}", ha='center', va='center',
                color='white' if weights_np[i,j] > 0.5 else 'black', fontsize=10, fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.046)

# ── Panel 2: Temperature effect — low temp (sharp) vs high temp (flat) ───
ax2 = axes[1]
# Simulate scores for a 5-token sequence
scores_5 = torch.tensor([2.1, 0.3, -0.5, 1.2, 0.8])
temps = [0.5, 1.0, 2.0]
colors = ['#e74c3c', '#2ecc71', '#3498db']
labels_t = ['T=0.5 (sharp)', 'T=1.0 (default)', 'T=2.0 (flat)']
for temp, color, label in zip(temps, colors, labels_t):
    w = F.softmax(scores_5 / temp, dim=0).numpy()
    ax2.plot(range(5), w, 'o-', color=color, label=label, linewidth=2, markersize=7)
ax2.set_xticks(range(5))
ax2.set_xticklabels([f"tok{i}" for i in range(5)], fontsize=9)
ax2.set_ylabel("Attention weight", fontsize=9)
ax2.set_title("Temperature Effect\non Softmax Distribution", fontsize=11)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)
ax2.axhline(0.2, linestyle='--', color='gray', alpha=0.5, label='uniform')

# ── Panel 3: Scaling effect (why divide by sqrt(d_k)) ────────────────────
ax3 = axes[2]
d_k_values = [1, 4, 16, 64, 256]
np.random.seed(42)
for dk in d_k_values:
    scores_raw = np.random.randn(dk).sum()   # dot product of random unit vectors
    # Show softmax entropy vs d_k to illustrate the problem
for dk in d_k_values:
    # Simulate 10 attention scores for different d_k
    sim_scores = np.random.randn(10) * np.sqrt(dk)   # scores grow with sqrt(d_k)
    w_unscaled  = torch.softmax(torch.tensor(sim_scores, dtype=torch.float32), dim=0).numpy()
    w_scaled    = torch.softmax(torch.tensor(sim_scores / np.sqrt(dk), dtype=torch.float32), dim=0).numpy()

# Bar plot comparing entropy for scaled vs unscaled
d_k_list = [4, 16, 64, 256, 1024]
entropy_unscaled = []
entropy_scaled   = []
for dk in d_k_list:
    sim_scores = np.random.randn(10) * np.sqrt(dk)
    w_u = torch.softmax(torch.tensor(sim_scores, dtype=torch.float32), dim=0).numpy()
    w_s = torch.softmax(torch.tensor(sim_scores / np.sqrt(dk), dtype=torch.float32), dim=0).numpy()
    entropy_unscaled.append(-np.sum(w_u * np.log(w_u + 1e-9)))
    entropy_scaled.append(-np.sum(w_s * np.log(w_s + 1e-9)))

x_pos = np.arange(len(d_k_list))
ax3.bar(x_pos - 0.2, entropy_unscaled, 0.4, label='Unscaled (too peaky)', color='#e74c3c', alpha=0.8)
ax3.bar(x_pos + 0.2, entropy_scaled,   0.4, label='Scaled ÷√d_k (stable)', color='#2ecc71', alpha=0.8)
ax3.set_xticks(x_pos)
ax3.set_xticklabels([f"d_k={dk}" for dk in d_k_list], fontsize=8)
ax3.set_ylabel("Attention entropy\n(higher = more distributed)", fontsize=9)
ax3.set_title("Why Scale by 1/√d_k?\nPrevents collapse to argmax", fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("images/attention_analysis.png", dpi=120, bbox_inches='tight')
plt.show()
print("Saved to images/attention_analysis.png")

---
## Step 4 — The MLP / Feed-Forward Block

After attention lets tokens communicate with each other, the **MLP** (also called FFN) processes each token **independently** — no cross-token communication here.

### Why does it exist?

Attention is a **routing/aggregation** mechanism. It moves information around.  
The MLP is a **computation** mechanism. It transforms the information.

A useful mental model:
- **Attention** = "retrieve relevant facts from other tokens"
- **MLP** = "think about what those facts mean for this token"

### The Math

```
Input:  x   shape [T, D] = [3, 768]

Linear up:    h = x @ W1 + b1      W1: [768, 3072]   → h: [T, 3072]
Activation:   h = GELU(h)          element-wise non-linearity
Linear down:  y = h @ W2 + b2      W2: [3072, 768]   → y: [T, 768]
```

### Why 4× expansion? (768 → 3072 → 768)

The 4× hidden dimension is where the "thinking" happens.  
Research shows models store **factual knowledge** in MLP weights.  
Larger expansion = more capacity to memorize facts.

### GELU vs ReLU

GPT-2 uses **GELU** (Gaussian Error Linear Unit) instead of ReLU:

```
ReLU(x) = max(0, x)           → hard cutoff at 0
GELU(x) = x · Φ(x)            → smooth, probabilistic gate
         where Φ = CDF of standard normal
```

GELU tends to work better for language models. The smooth curve helps gradient flow.

### After the MLP: Residual Connection

```
x = x + MLP(LayerNorm(x))
```

The original `x` is added back (residual). This means:
- The MLP only needs to learn the **delta** (improvement), not the full representation
- Gradients can flow directly from the output all the way back to the input
- This is why we can stack 12 (or 96 in GPT-4) layers without exploding gradients

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 4: MLP Block — GELU activation and shape walkthrough
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("STEP 4: MLP / Feed-Forward Block")
print("=" * 60)

def gelu(x):
    """GPT-2's exact GELU approximation (matches HuggingFace implementation)"""
    return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)))

# ── Plot GELU vs ReLU ─────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
xs = torch.linspace(-3, 3, 300)
ax.plot(xs, torch.relu(xs), 'r-', linewidth=2, label='ReLU: max(0,x)')
ax.plot(xs, gelu(xs), 'b-', linewidth=2, label='GELU: x·Φ(x)')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlabel("x"); ax.set_ylabel("activation(x)")
ax.set_title("GELU vs ReLU — GPT-2 uses GELU for smoother gradients", fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.5, 3)
plt.tight_layout()
plt.savefig("images/gelu_vs_relu.png", dpi=120, bbox_inches='tight')
plt.show()

# ── MLP forward pass with exact shapes ────────────────────────────────────
print("\nMLP forward pass (GPT-2 Small dimensions):")
torch.manual_seed(5)

# Input: residual stream after attention sublayer (3 tokens × 768 dims)
x_in = torch.randn(3, 768)   # [T, D_MODEL]

W1 = torch.randn(768, 3072) * 0.02   # [D_MODEL, D_FF]
b1 = torch.zeros(3072)
W2 = torch.randn(3072, 768) * 0.02   # [D_FF, D_MODEL]
b2 = torch.zeros(768)

print(f"\n  x_in  shape: {x_in.shape}    (T=3 tokens, D=768)")
h = x_in @ W1 + b1
print(f"  h     shape: {h.shape}   after W1 (linear up: 768→3072)")
h_act = gelu(h)
print(f"  GELU(h) shape: {h_act.shape}  (same, just element-wise non-linearity)")
y = h_act @ W2 + b2
print(f"  y     shape: {y.shape}   after W2 (linear down: 3072→768)")

# Residual connection
x_out = x_in + y
print(f"  x_out shape: {x_out.shape}   after residual: x_in + MLP(x_in)")

print(f"\nParameter count in this MLP:")
print(f"  W1: {768 * 3072:,} params")
print(f"  b1: {3072:,} params")
print(f"  W2: {3072 * 768:,} params")
print(f"  b2: {768:,} params")
mlp_params = 768*3072 + 3072 + 3072*768 + 768
print(f"  Total per layer: {mlp_params:,} params")
print(f"  × 12 layers: {mlp_params * 12:,} params in all MLPs ({mlp_params*12/1e6:.1f}M)")

---
## Step 5 — From Logits to Words: The Output Stage

After 12 transformer blocks, we have a refined residual stream of shape `[T, 768]`.  
The final step converts this back to vocabulary probabilities.

### The Unembedding Step

```
1. Final LayerNorm:
   x_final = LayerNorm(x_after_12_layers)    shape: [T, 768]

2. Unembed (project to vocab):
   logits = x_final @ W_E.T                  shape: [T, 50257]
   
   W_E.T is the TRANSPOSE of the token embedding matrix (weight tying)
   GPT-2 reuses the same weights for embedding and unembedding
   This cuts ~39M params and improves generalization

3. We only care about the LAST token's logits (for next-token prediction):
   logits_last = logits[-1]                   shape: [50257]

4. Softmax → probabilities:
   probs = softmax(logits_last / temperature) shape: [50257]
```

### Weight Tying — Why Reuse Embedding Weights?

The same matrix `W_E` [50257, 768] is used:
- **Forward pass**: look up a row to get a token's embedding
- **Backward pass (unembed)**: multiply by its transpose to get logit scores

The intuition: if the model embeds "Paris" close to "capital", it should also assign  
high probability to "Paris" when "capital" is in the context.

### Softmax Temperature in Generation

```python
temperature = 0.8   # typical for creative generation
probs = softmax(logits / temperature)

temperature < 1.0 → sharpens distribution → model is more confident/deterministic
temperature > 1.0 → flattens distribution → model is more random/creative
temperature = 1.0 → default, no change
```

### Top-k and Top-p Sampling

Even after softmax, we don't always pick the argmax (greedy decoding).  
Two common strategies:

```
Top-k sampling: keep only the k highest-probability tokens, renormalize, sample
  e.g. k=50: consider only the top 50 tokens

Top-p (nucleus) sampling: keep the smallest set of tokens whose cumulative prob ≥ p
  e.g. p=0.9: might keep 20 tokens in some steps, 200 in others
```

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# STEP 5: Output Stage — logits, softmax, temperature, top-k/top-p sampling
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("STEP 5: From Hidden States → Logits → Next Token")
print("=" * 60)

torch.manual_seed(42)
VOCAB_SIZE = 50257
D_MODEL    = 768

# Simulated final hidden state for "the cat sat" (3 tokens × 768 dims)
x_final = torch.randn(3, D_MODEL)

# Final LayerNorm
ln_final = nn.LayerNorm(D_MODEL)
x_normed = ln_final(x_final)
print(f"\n1. Final LayerNorm  →  x_normed shape: {x_normed.shape}")

# Unembed (weight-tied: use same W_E transposed)
# In practice W_E is shared with the embedding layer
W_E_sim = torch.randn(VOCAB_SIZE, D_MODEL) * 0.02
logits = x_normed @ W_E_sim.T     # [T, 50257]
print(f"2. Unembed: x_normed @ W_E.T  →  logits shape: {logits.shape}")

# Take LAST token's logits (predicting what comes after "sat")
logits_last = logits[-1]           # [50257]
print(f"3. Take last token logits  →  shape: {logits_last.shape}")

# ── Greedy Decoding (argmax) ──────────────────────────────────────────────
greedy_token = logits_last.argmax().item()
greedy_prob  = F.softmax(logits_last, dim=0)[greedy_token].item()
print(f"\nGreedy prediction: token_id={greedy_token}  (prob={greedy_prob:.4f})")

# ── Top-5 tokens ──────────────────────────────────────────────────────────
probs = F.softmax(logits_last, dim=0)
top5_probs, top5_ids = probs.topk(5)
print(f"\nTop-5 predicted tokens:")
print(f"  {'Rank':<6} {'token_id':<12} {'probability':<14}")
print(f"  {'-'*32}")
for rank, (tid, prob) in enumerate(zip(top5_ids.tolist(), top5_probs.tolist()), 1):
    print(f"  {rank:<6} {tid:<12} {prob:.6f}")

# ── Temperature effect ────────────────────────────────────────────────────
print(f"\nEffect of temperature on top-5 probabilities:")
print(f"  {'Temperature':<14} {'Entropy (bits)':<18} {'Top-1 prob'}")
print(f"  {'-'*46}")
for temp in [0.3, 0.7, 1.0, 1.5, 2.0]:
    p = F.softmax(logits_last / temp, dim=0)
    entropy = -(p * p.log2().clamp(min=-100)).sum().item()
    top1 = p.max().item()
    note = " ← greedy/sharp" if temp < 0.5 else (" ← default" if temp == 1.0 else " ← creative/flat" if temp > 1.2 else "")
    print(f"  {temp:<14.1f} {entropy:<18.3f} {top1:.6f}{note}")

# ── Top-k Sampling demo ────────────────────────────────────────────────────
def top_k_sample(logits, k=50, temperature=1.0):
    """Filter to top-k tokens, then sample"""
    scaled = logits / temperature
    topk_vals, topk_ids = scaled.topk(k)
    probs = F.softmax(topk_vals, dim=0)
    chosen_idx = torch.multinomial(probs, 1).item()
    return topk_ids[chosen_idx].item()

# ── Top-p (nucleus) Sampling demo ─────────────────────────────────────────
def top_p_sample(logits, p=0.9, temperature=1.0):
    """Filter to smallest set with cumulative prob ≥ p, then sample"""
    scaled = logits / temperature
    sorted_probs, sorted_ids = F.softmax(scaled, dim=0).sort(descending=True)
    cumulative = sorted_probs.cumsum(dim=0)
    # Find cutoff: keep tokens until cumulative prob exceeds p
    cutoff = (cumulative < p).sum().item() + 1
    nucleus_probs = sorted_probs[:cutoff]
    nucleus_ids   = sorted_ids[:cutoff]
    probs_renorm  = nucleus_probs / nucleus_probs.sum()
    chosen_idx    = torch.multinomial(probs_renorm, 1).item()
    return nucleus_ids[chosen_idx].item()

print(f"\nSampling strategies (5 samples each, for illustration):")
print(f"  Greedy  (always same): {[greedy_token]*5}")
k50_samples = [top_k_sample(logits_last, k=50) for _ in range(5)]
p90_samples = [top_p_sample(logits_last, p=0.9) for _ in range(5)]
print(f"  Top-k=50  (diverse):  {k50_samples}")
print(f"  Top-p=0.9 (nucleus):  {p90_samples}")

print("""
Summary:
  - Greedy = deterministic, often repetitive
  - Top-k  = diverse but may include low-quality tokens
  - Top-p  = adaptive: narrow when model is confident, wider when uncertain
  - Most production systems use top-p + temperature together
""")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# FINAL: Complete GPT-2-style Transformer — clean, production-quality implementation
# Everything we've learned, assembled into one readable module
# ──────────────────────────────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention (GPT-2 style).
    
    Key decisions:
    - Pre-norm (LayerNorm BEFORE attention, not after)
    - Causal mask (lower-triangular, registered as buffer)
    - Weight-tied Q/K/V via single combined projection (3x faster)
    - Output projection W_O maps concat(heads) back to d_model
    """
    def __init__(self, d_model, n_heads, max_ctx=1024, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads

        # Combined Q+K+V projection (one matmul instead of three)
        self.c_attn  = nn.Linear(d_model, 3 * d_model)
        self.c_proj  = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        # Causal mask: lower-triangular matrix registered as buffer (not trained)
        mask = torch.tril(torch.ones(max_ctx, max_ctx))
        self.register_buffer("mask", mask.view(1, 1, max_ctx, max_ctx))

    def forward(self, x):
        B, T, D = x.shape

        # Project to Q, K, V all at once then split
        qkv = self.c_attn(x)                      # [B, T, 3*D]
        Q, K, V = qkv.split(D, dim=2)             # each [B, T, D]

        # Reshape for multi-head: [B, T, D] → [B, n_heads, T, d_head]
        def reshape(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        Q, K, V = reshape(Q), reshape(K), reshape(V)

        # Scaled dot-product attention
        scale  = 1.0 / math.sqrt(self.d_head)
        scores = Q @ K.transpose(-2, -1) * scale  # [B, n_heads, T, T]

        # Apply causal mask (set future positions to -inf)
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

        # Softmax → weights → weighted sum of values
        w = F.softmax(scores, dim=-1)
        w = self.dropout(w)
        out = w @ V                                # [B, n_heads, T, d_head]

        # Concatenate heads: [B, n_heads, T, d_head] → [B, T, D]
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.c_proj(out)


class MLP(nn.Module):
    """
    Position-wise feed-forward network (GPT-2 style).
    768 → 3072 (GELU) → 768
    Each token processed INDEPENDENTLY — no communication between tokens here.
    """
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.c_fc   = nn.Linear(d_model, 4 * d_model)
        self.c_proj = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def gelu(self, x):
        return 0.5 * x * (1.0 + torch.tanh(math.sqrt(2.0/math.pi) * (x + 0.044715 * x**3)))

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    """
    One complete transformer block:
      x = x + Attn(LN(x))   ← attention sublayer with residual
      x = x + MLP(LN(x))    ← MLP sublayer with residual
    
    The residual stream x flows through unchanged — each sublayer adds a delta.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ln_2 = nn.LayerNorm(d_model)
        self.mlp  = MLP(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # attention sublayer
        x = x + self.mlp(self.ln_2(x))    # MLP sublayer
        return x


class GPT2(nn.Module):
    """
    GPT-2 style language model.
    
    Architecture:
      token_emb + pos_emb → 12 × TransformerBlock → LayerNorm → unembed → logits
    
    Weight tying: input embedding matrix W_E is shared with output unembedding.
    """
    def __init__(self, vocab_size=50257, d_model=768, n_heads=12,
                 n_layers=12, max_ctx=1024, dropout=0.1):
        super().__init__()
        self.d_model   = d_model
        self.max_ctx   = max_ctx

        # Token + positional embeddings
        self.wte = nn.Embedding(vocab_size, d_model)    # W_E: token embedding
        self.wpe = nn.Embedding(max_ctx, d_model)       # W_P: positional embedding
        self.drop = nn.Dropout(dropout)

        # 12 transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)
        ])

        # Final layer norm
        self.ln_f = nn.LayerNorm(d_model)

        # Output head (weight-tied to wte — no extra params)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight   # ← weight tying

        # Initialize weights
        self.apply(self._init_weights)
        # Scale residual projections (GPT-2 paper: 1/sqrt(n_layers))
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        """
        idx: token IDs, shape [B, T]
        returns logits of shape [B, T, vocab_size]
        """
        B, T = idx.shape
        assert T <= self.max_ctx, f"Sequence length {T} exceeds max_ctx {self.max_ctx}"

        # Build input: token embedding + positional embedding
        positions = torch.arange(T, device=idx.device)    # [T]
        x = self.drop(self.wte(idx) + self.wpe(positions))  # [B, T, D]

        # Pass through all transformer blocks (residual stream updated each block)
        for block in self.blocks:
            x = block(x)

        # Final LayerNorm + project to vocab
        x = self.ln_f(x)                   # [B, T, D]
        logits = self.lm_head(x)           # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=20, temperature=1.0, top_k=50):
        """
        Autoregressive generation: repeatedly predict next token and append it.
        idx: starting token IDs, shape [1, T]
        """
        for _ in range(max_new_tokens):
            # Crop to max_ctx if needed
            idx_ctx = idx[:, -self.max_ctx:]
            # Get logits for last position
            logits = self(idx_ctx)[:, -1, :] / temperature   # [1, vocab_size]
            # Top-k filtering
            if top_k is not None:
                v, _ = logits.topk(top_k)
                logits[logits < v[:, -1:]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # [1, 1]
            idx = torch.cat([idx, next_token], dim=1)             # [1, T+1]
        return idx


# ──────────────────────────────────────────────────────────────────────────────
# Instantiate the model and print parameter breakdown
# ──────────────────────────────────────────────────────────────────────────────

# Use "nano" config for quick demo (same architecture, smaller dims)
nano_config = dict(vocab_size=50257, d_model=128, n_heads=4, n_layers=4,
                   max_ctx=256, dropout=0.0)

model = GPT2(**nano_config)

print("GPT-2 Nano (demo config) — Parameter Breakdown")
print("=" * 55)
total = 0
for name, param in model.named_parameters():
    if 'lm_head' in name:
        continue   # skip — weight-tied, already counted in wte
    n = param.numel()
    total += n
    print(f"  {name:<35} {n:>10,}   shape={list(param.shape)}")

print(f"\n  Total (excluding tied lm_head): {total:,} params")
print(f"  Full GPT-2 Small (d=768, 12L): ~85,000,000 params")

# ── Quick forward pass ────────────────────────────────────────────────────
dummy_input = torch.randint(0, 50257, (1, 10))   # batch=1, seq_len=10
logits = model(dummy_input)
print(f"\nForward pass:")
print(f"  Input  shape: {dummy_input.shape}      (batch=1, T=10 tokens)")
print(f"  Output shape: {logits.shape}  (batch=1, T=10, vocab=50257)")
print(f"  → Pick last position logits: {logits[:, -1, :].shape}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY: Complete architecture diagram as ASCII + parameter math
# ──────────────────────────────────────────────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════════════╗
║          GPT-2 Small — Complete Forward Pass Diagram                 ║
║          Input: "the cat sat"  (T=3 tokens)                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  ["the", "cat", "sat"]                                               ║
║         │                                                            ║
║         ▼ tokenize                                                   ║
║  [262, 3797, 3332]          shape: [T=3]                            ║
║         │                                                            ║
║         ├── W_E lookup ──► token_emb  [T, 768]  50257×768 matrix   ║
║         ├── W_P lookup ──► pos_emb    [T, 768]  1024×768 matrix    ║
║         └──────────────────────────────────────────────             ║
║                           x₀ = token_emb + pos_emb  [T, 768]       ║
║                                      │                              ║
║         ╔════════════════════════════╧═══════════╗                  ║
║         ║  TRANSFORMER BLOCK ×12                 ║                  ║
║         ║                                        ║                  ║
║         ║  x = x + Attn(LayerNorm₁(x))          ║                  ║
║         ║       │                                ║                  ║
║         ║       ├── LN₁: normalize [T,768]       ║                  ║
║         ║       ├── Q=xW_Q [T,64] per head       ║                  ║
║         ║       ├── K=xW_K [T,64] per head       ║                  ║
║         ║       ├── V=xW_V [T,64] per head       ║                  ║
║         ║       ├── scores = QKᵀ/√64  [12,T,T]  ║                  ║
║         ║       ├── causal mask (future=-∞)       ║                  ║
║         ║       ├── softmax → weights [12,T,T]    ║                  ║
║         ║       ├── output = weights@V [T,768]    ║                  ║
║         ║       └── residual add back to x        ║                  ║
║         ║                                        ║                  ║
║         ║  x = x + MLP(LayerNorm₂(x))            ║                  ║
║         ║       │                                ║                  ║
║         ║       ├── LN₂: normalize [T,768]       ║                  ║
║         ║       ├── FC₁: [T,768] → [T,3072]      ║                  ║
║         ║       ├── GELU activation               ║                  ║
║         ║       ├── FC₂: [T,3072] → [T,768]      ║                  ║
║         ║       └── residual add back to x        ║                  ║
║         ╚════════════════════════════════════════╝                  ║
║                                      │                              ║
║                           x₁₂ = final residual  [T, 768]           ║
║                                      │                              ║
║                           LayerNorm_f(x₁₂)       [T, 768]          ║
║                                      │                              ║
║                           @ W_E.T (weight-tied)                     ║
║                                      │                              ║
║                           logits               [T, 50257]           ║
║                                      │                              ║
║                           softmax(logits[-1])  [50257]              ║
║                                      │                              ║
║                           sample → next token ID → next word        ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# ── Parameter count breakdown (GPT-2 Small exact) ────────────────────────
print("Parameter Count — GPT-2 Small (exact)")
print("=" * 50)

vocab  = 50257
d      = 768
n_h    = 12
d_h    = 64
d_ff   = 3072
n_l    = 12
ctx    = 1024

wte_params = vocab * d
wpe_params = ctx * d

# Per-layer attention params
attn_qkv   = 3 * d * d   # combined Q+K+V (weight)
attn_qkv_b = 3 * d       # bias
attn_proj  = d * d
attn_proj_b = d
attn_total = attn_qkv + attn_qkv_b + attn_proj + attn_proj_b

# Per-layer MLP params
mlp_fc1   = d * d_ff;  mlp_fc1_b = d_ff
mlp_fc2   = d_ff * d;  mlp_fc2_b = d
mlp_total = mlp_fc1 + mlp_fc1_b + mlp_fc2 + mlp_fc2_b

# Per-layer layer norm (2 per block)
ln_params = 2 * 2 * d   # 2 LNs × (γ + β) × d

per_layer = attn_total + mlp_total + ln_params
all_layers = per_layer * n_l

# Final LN
ln_final_p = 2 * d

# Total (lm_head is weight-tied, not counted separately)
total_params = wte_params + wpe_params + all_layers + ln_final_p

print(f"  Token embeddings  (W_E): {wte_params:>12,}")
print(f"  Position embs     (W_P): {wpe_params:>12,}")
print(f"  Per transformer block:   {per_layer:>12,}")
print(f"    ├─ Attention (Q+K+V+O): {attn_total:>10,}")
print(f"    ├─ MLP (FC1+FC2):       {mlp_total:>10,}")
print(f"    └─ LayerNorms (×2):     {ln_params:>10,}")
print(f"  × 12 layers:             {all_layers:>12,}")
print(f"  Final LayerNorm:         {ln_final_p:>12,}")
print(f"  ─────────────────────────────────────")
print(f"  TOTAL (weight-tied):     {total_params:>12,}  ({total_params/1e6:.1f}M)")
print()
print("Key ratios:")
print(f"  Attention params / layer:  {attn_total/per_layer*100:.1f}%")
print(f"  MLP params / layer:        {mlp_total/per_layer*100:.1f}%")
print(f"  Embeddings / total:        {(wte_params+wpe_params)/total_params*100:.1f}%")